# CPTAC 归一化复评：输入准备核对

## tl;dr

942 个病例的原始计数与旧记录一致，旧 CPM 输入精确重建。两种新 size factor 的独立计算误差小于 2×10⁻¹⁴。这里只验收输入准备，不报告尚未核验的检索表现。

## Context & Methods

### Key Assumptions

沿用全部 30 个问题和原候选池。942 个文库联合拟合，因此属于传导式预处理，不是训练集冻结的单样本接口。原结果已被观察，本轮不是确认性试验。完整内部协议保留在 SSD，公开副本去掉了含病例文件名的清单，并保留其摘要和原协议哈希。

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
protocol = json.loads((root / 'PROTOCOL.public.json').read_text())
summary = json.loads((root / 'preparation_summary.public.json').read_text())
assert summary['samples'] == 942
assert summary['all_raw_columns_match_old_QC_hashes']
assert summary['old_CPM_reconstruction_exact']
assert not summary['qrel_values_read']
assert summary['old_files_unchanged']
print('输入一致性、旧流程重建和无标签准备检查通过。')

输入一致性、旧流程重建和无标签准备检查通过。


## Data

固定映射后的基因覆盖，不将未映射与真实零表达混淆。

In [2]:
pd.DataFrame([{'病例数': summary['samples'], '映射参考基因数': summary['mapped_reference_genes'], '可测选定基因数': summary['measured_selected_genes'], '缺失选定基因数': 4096-summary['measured_selected_genes']}])

,病例数,映射参考基因数,可测选定基因数,缺失选定基因数
0,942,23628,3502,594


## Results

表中为输入准备的汇总，不是模型有效性结果。

In [3]:
statistics = pd.DataFrame(summary['summaries']).T
assert (statistics['independent_factor_max_abs_error'] < 2e-14).all()
assert (statistics['factor_min'] > 0).all()
statistics[['fit_genes', 'positive_in_all_samples_genes', 'factor_min', 'factor_median', 'factor_max', 'independent_factor_max_abs_error']]

,fit_genes,positive_in_all_samples_genes,factor_min,factor_median,factor_max,independent_factor_max_abs_error
sf_reference,23628.0,13402.0,0.184351,1.076751,1.981186,1.865175e-14
sf_full_measured,59808.0,14370.0,0.183958,1.080050,1.995303,1.909584e-14


## Takeaways

输入准备已通过；仍须完成全部推断、共有可测蛋白池的公平比较和独立指标重建。保留所有旧条件、新条件和阴性结果；不把技术一致性当作生物学有效性或 Cell 投稿准备完成。完整命令和缺失依赖边界见 README。